# 5장 스킬

에이전트가 도구를 호출해 외부 정보를 가져오고 작업을 수행합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 5장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

---

## 1. 언어 모델 선택

스킬은 도구를 호출하듯 활성화할 수 있습니다. 프롬프트로 도구를 호출하는 Gemma 3와 내장 도구 호출을 사용하는 Gemma 4 모두 이 방식으로 스킬을 쓸 수 있습니다. 여기서는 뒤의 실습에 맞춰 Gemma 4를 선택합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B(내장 추론과 도구 호출 지원)
llm = LLM(model="gemma4:e4b")

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. `SKILL.md`로 작업 절차 정의

도구와 MCP는 에이전트가 수행할 수 있는 동작을 제공합니다. `SKILL.md`에는 특정 작업에서 어떤 동작을 어떤 순서로 수행할지 적습니다. 발표 자료를 만드는 경우에는 검색 도구로 자료를 찾고, 모델이 내용을 요약한 뒤, 슬라이드 생성 도구로 한 장씩 작성하는 순서를 지시할 수 있습니다.

파일 맨 위의 YAML에는 스킬 이름과 짧은 설명을 적습니다. 이 부분은 항상 모델의 입력에 포함됩니다.

```yaml
name: ...
description: ...
```

아래 마크다운 본문에는 자세한 작업 절차와 필요한 분야 지식을 적습니다.

```markdown
---
name: ...
description: ...
---

# 작업 이름

## 실행 절차
...
```

반복해서 지정해야 하는 문체나 데이터베이스 스키마를 스킬에 담아 둘 수 있습니다. 시스템 프롬프트에는 처음부터 이름과 설명만 넣고, 스킬을 활성화하는 시점에 전체 지침을 읽습니다. 이렇게 필요한 지침을 단계적으로 제공하면 쓰지 않는 스킬이 모델의 입력 공간을 차지하는 양을 줄일 수 있습니다.

`Skills`는 `Tools`를 상속합니다. 에이전트는 일반 도구와 같은 인터페이스로 스킬을 선택하고 지침을 읽을 수 있습니다.

In [ ]:
import inspect
import yaml
from pathlib import Path
from illustrated_agents.chapters.ch5 import NativeTools

class Skills(NativeTools):
    """도구와 스킬을 함께 관리한다.

    스킬은 특정 작업을 어떻게 수행할지 설명한다.
    스킬을 활성화하면 프로그램이 작업 지침을 반환한다.
    계산 도구와 스킬이 반환하는 값은 다르지만
    에이전트가 사용할 시점을 선택하는 방식은 같다.

    스킬의 내용은 필요한 시점에 읽는다.
    시스템 프롬프트에는 이름과 설명을 제공하고
    에이전트가 스킬을 호출하면 전체 지침을 전달한다.
    """

    def add_skill(self, path: str):
        """스킬 폴더에서 SKILL.md를 읽어 도구로 등록한다."""
        skill_dir = Path(path)
        content = (skill_dir / "SKILL.md").read_text(encoding="utf-8")
        
        # 2단계: SKILL.md 본문의 전체 지침
        _, frontmatter, instructions = content.split("---", 2)
        meta = yaml.safe_load(frontmatter)

        # 1단계: 스킬 목록에 표시할 이름과 설명
        name, description = meta["name"], meta["description"]
        
        # 3단계: 같은 폴더에서 에이전트가 이용할 수 있는 파일 목록
        extra_files = [p.name for p in skill_dir.iterdir() if p.name != "SKILL.md"]
        
        # 활성화하면 본문 지침과 폴더 정보를 반환
        def skill(**kwargs):
            return f"""{instructions}

Skill folder: {skill_dir}

Available files: {extra_files}"""
        
        skill.__name__ = name
        skill.__doc__ = "A skill. " + description
        skill.__signature__ = inspect.Signature()
        self.add_tool(name, skill, "A skill. " + description)

    @property
    def prompt(self):
        return """You have specialized skills available.
To activate a skill, call it with an empty argument object.
Skills return instructions; they do not process the task directly."""

스킬을 추가하는 메서드와, 등록한 스킬을 모델에 알려 주는 프롬프트를 살펴봅니다.

[파일 분석 스킬](../src/illustrated_agents/skills/file_analyzer/SKILL.md)에는 `read_markdown`으로 문서를 읽고 내용을 요약하는 순서가 들어 있습니다. 준비된 스킬을 불러와 내용을 확인합니다.

In [ ]:
import illustrated_agents

# 파일 분석 스킬을 예제로 선택
file_analyzer_path = Path(illustrated_agents.__file__).parent / "skills" / "file_analyzer"

# 스킬 불러오기
tools_and_skills = Skills()
tools_and_skills.add_skill(file_analyzer_path)

에이전트에 먼저 전달되는 짧은 설명을 확인합니다.

In [ ]:
tools_and_skills.registry["file_analyzer"]["description"]

짧은 설명에는 작업의 용도만 나옵니다. 아래처럼 스킬을 호출하면 자세한 실행 지침을 읽을 수 있습니다.

In [ ]:
tools_and_skills.prompt

In [ ]:
from illustrated_agents.chapters.ch2 import Response

response = Response(
  content="",
  tool_call={"function": {"name": "file_analyzer", "arguments": {}}},
)

response = tools_and_skills.parse(response)
print(tools_and_skills.execute(response))

이 정도 길이의 지침을 스킬마다 시스템 프롬프트에 넣으면 사용 가능한 컨텍스트가 빠르게 줄어듭니다.

## 3. `TinyAgent` 실행

스킬을 기존 도구처럼 호출하므로 `TinyAgent`를 수정하지 않고 연결할 수 있습니다.

In [ ]:
import requests
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import TinyAgent

def read_markdown(path: str) -> str:
    """Read the content of a markdown file.

    Args:
        path (str): The path to the markdown file.
    """
    return requests.get(path).text

# 도구와 스킬
tools_and_skills = Skills()
tools_and_skills.add_skill(file_analyzer_path)
tools_and_skills.add_tool("read_markdown", read_markdown)

# 에이전트 생성
agent = TinyAgent(
    llm=llm,
    tools=tools_and_skills,
    memory=Memory(),
)

In [ ]:
query = "Use the `file_analyzer` skill to summarize https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/refs/heads/main/README.md."

print(agent.run(query))

위에서 `file_analyzer`를 호출해 얻은 관찰 결과는 `SKILL.md`의 내용입니다. 실제 분석을 마치려면 에이전트가 그 지침을 읽고 다음 도구를 호출해야 합니다. 이번 장의 에이전트는 한 번만 호출하므로 이후 단계를 자동으로 수행하는 방법은 6장에서 구현합니다.

---

# 구현한 내용

스킬을 등록하고 필요할 때 지침 전문을 읽어 에이전트에 작업 절차를 전달했습니다.

In [ ]:
from illustrated_agents.chapters.ch5 import what_we_built_skills; what_we_built_skills

# 다음 장

[6장](../chapter06/chapter06.ipynb)에서는 에이전트가 도구 결과를 확인하고 다음 행동을 정하는 ReAct 반복문을 구현합니다.